# Cicero Digital – Kapitel 5: Netzwerke

In diesem Notebook bauen wir **zwei einfache Netzwerke**:

1. ein **Brief-Netzwerk auf Basis von TF-IDF-Ähnlichkeiten**
2. ein **Personen-Netzwerk** aus **Absender**, **Empfänger** und **Datierung**

Danach ergänzen wir das Personen-Netzwerk um eine **Zeitdimension**.  
Wenn Topic-Zuweisungen aus Kapitel 4 vorhanden sind, nehmen wir zusätzlich das **wahrscheinlichste Topic** pro Brief dazu.

Der Code ist bewusst **so einfach wie möglich** gehalten.

## 0. Setup

Wir verwenden vor allem:

- `pandas` für Tabellen
- `networkx` für Netzwerke
- `matplotlib` für einfache Visualisierungen
- `scikit-learn` für TF-IDF und Kosinus-Ähnlichkeiten

Für den zeitlichen Teil gibt es zusätzlich einen **optional**en Abschnitt mit `pathpyG`.

In [ ]:
# Falls nötig, Pakete installieren:
# !pip install pandas numpy matplotlib seaborn networkx scikit-learn
# Optional für den zeitlichen Teil:
# !pip install torch torch_geometric git+https://github.com/pathpy/pathpyG.git

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import ipywidgets as widgets
from IPython.display import display, clear_output

from collections import Counter
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams["figure.figsize"] = (10, 7)

## 1. Daten laden

Wir gehen davon aus, dass das Brief-Dataset als CSV vorliegt.

Optional laden wir zusätzlich die Datei aus Kapitel 4 mit den Topic-Zuweisungen.  
Wenn diese Datei fehlt, funktioniert das Notebook trotzdem.

In [ ]:
base_dir = Path("outputs")
csv_path = base_dir / "cicero_letters.csv"
topic_path = base_dir / "topic_modeling" / "letters_with_topics.csv"

df = pd.read_csv(csv_path)
print("Briefe insgesamt:", len(df))
df.head(3)

In [ ]:
print(df.columns.tolist())

## 2. Hilfsfunktionen

Wir brauchen drei kleine Hilfsfunktionen:

- eine für eine einfache `doc_id`
- eine für eine sehr vorsichtige Jahres-Extraktion
- eine für eine einfache Textbereinigung

In [ ]:
def make_doc_id(row, idx):
    corpus = str(row.get("corpus", "unknown")).strip()
    book_n = row.get("book_n", "x")
    letter_n = row.get("letter_n", idx)

    if pd.isna(book_n):
        book_n = "x"
    if pd.isna(letter_n):
        letter_n = idx

    return f"{corpus}_b{book_n}_l{letter_n}_r{idx}"


def extract_year(row):
    # 1. Falls es bereits eine Year-Spalte gibt, bevorzugen wir diese
    if "year" in row and pd.notna(row["year"]):
        value = str(row["year"]).strip()
        m = re.search(r"-?\d{1,4}", value)
        if m:
            try:
                return int(m.group())
            except Exception:
                pass

    # 2. Sonst versuchen wir date_when
    if "date_when" in row and pd.notna(row["date_when"]):
        value = str(row["date_when"]).strip()
        m = re.search(r"-?\d{1,4}", value)
        if m:
            try:
                return int(m.group())
            except Exception:
                pass

    return np.nan


LATIN_STOPWORDS = {
    "et", "in", "de", "ad", "non", "ut", "cum", "qui", "quae", "quod",
    "est", "esse", "sum", "ego", "tu", "nos", "vos", "hic", "haec", "hoc",
    "ille", "illa", "illud", "is", "ea", "id", "autem", "enim", "sed",
    "si", "ne", "nec", "nam", "ita", "iam", "me", "te", "se", "mihi",
    "tibi", "sibi", "quo", "quod", "quam", "atque"
}


def simple_clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    tokens = [t for t in text.split() if len(t) > 2 and t not in LATIN_STOPWORDS]
    return " ".join(tokens)

In [ ]:
df = df.copy()
df["doc_id"] = [make_doc_id(row, idx) for idx, (_, row) in enumerate(df.iterrows())]
df["year_simple"] = df.apply(extract_year, axis=1)
df["clean_text"] = df["text"].fillna("").map(simple_clean_text)

show_cols = [c for c in ["doc_id", "corpus", "book_n", "letter_n", "sender", "recipient", "date_when", "year", "year_simple"] if c in df.columns]
df[show_cols].head()

## 3. Topics aus Kapitel 4 optional dazunehmen

Wenn die Datei `letters_with_topics.csv` vorhanden ist, übernehmen wir daraus pro Brief:

- `topic`
- `topic_prob`

Das reicht für Kapitel 5 völlig aus.

ACHTUNG: Folgende Zelle nur einmal ausführen! Wenn mehrmals ausgeführt, kann es zu Fehlern kommen. Für diesen Fall das Notebook nochmals von vorne ausführen und `df` neu konstruieren.

In [ ]:
if topic_path.exists():
    topics_df = pd.read_csv(topic_path)

    keep_cols = [c for c in ["doc_id", "dominant_topic", "topic_prob", "topic_keywords"] if c in topics_df.columns]
    topics_df = topics_df[keep_cols].drop_duplicates("doc_id")

    df = df.merge(topics_df, on="doc_id", how="left")
    print("Topic-Datei gefunden.")
else:
    df["topic"] = np.nan
    df["topic_prob"] = np.nan
    print("Keine Topic-Datei gefunden. Wir arbeiten ohne Topics.")

df[[c for c in ["doc_id", "dominant_topic", "topic_prob"] if c in df.columns]].head()

## 4. Teil A – Einfaches TF-IDF-Ähnlichkeitsnetzwerk

Die Idee ist sehr simpel:

- Jeder Brief wird als TF-IDF-Vektor dargestellt.
- Zwischen zwei Briefen berechnen wir die **Kosinus-Ähnlichkeit**.
- Wenn die Ähnlichkeit hoch genug ist, verbinden wir die Briefe mit einer Kante.

Wichtig: Für ein Lehr-Notebook halten wir das absichtlich klein und einfach.

In [ ]:
tfidf_df = df.copy()

# Nur Briefe mit ein wenig Text verwenden
tfidf_df = tfidf_df[tfidf_df["clean_text"].str.split().str.len() >= 5].copy()

# Für eine erste, übersichtliche Netzwerk-Grafik beschränken wir die Zahl der Dokumente
max_docs = 80
tfidf_df = tfidf_df.head(max_docs).copy()

print("Dokumente im TF-IDF-Teil:", len(tfidf_df))
tfidf_df[["doc_id", "clean_text"]].head(3)

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(tfidf_df["clean_text"])
similarity = cosine_similarity(X)

sim_df = pd.DataFrame(similarity, index=tfidf_df["doc_id"], columns=tfidf_df["doc_id"])
sim_df.iloc[:8, :8]

### Ein sehr einfaches Netzwerk bauen

Wir setzen einen **Schwellenwert**.  
Nur wenn zwei Briefe ähnlich genug sind, fügen wir eine Kante ein.

In [ ]:
threshold = 0.25

G_tfidf = nx.Graph()

for doc_id in tfidf_df["doc_id"]:
    G_tfidf.add_node(doc_id)

doc_ids = tfidf_df["doc_id"].tolist()

for i in range(len(doc_ids)):
    for j in range(i + 1, len(doc_ids)):
        score = similarity[i, j]
        if score >= threshold:
            G_tfidf.add_edge(doc_ids[i], doc_ids[j], weight=score)

print("Knoten:", G_tfidf.number_of_nodes())
print("Kanten:", G_tfidf.number_of_edges())

In [ ]:
# Kleine Netzwerk-Übersicht
degrees = dict(G_tfidf.degree())

deg_df = (
    pd.DataFrame({
        "doc_id": list(degrees.keys()),
        "degree": list(degrees.values())
    })
    .sort_values("degree", ascending=False)
)

deg_df.head(10)

In [ ]:
# Nur Knoten mit mindestens einer Kante plotten
nodes_with_edges = [n for n, d in G_tfidf.degree() if d > 0]
G_tfidf_plot = G_tfidf.subgraph(nodes_with_edges).copy()

plt.figure(figsize=(12, 10))

if len(G_tfidf_plot) > 0:
    pos = nx.spring_layout(G_tfidf_plot, seed=42)
    weights = [G_tfidf_plot[u][v]["weight"] * 3 for u, v in G_tfidf_plot.edges()]

    nx.draw_networkx_nodes(G_tfidf_plot, pos, node_size=120, alpha=0.8)
    nx.draw_networkx_edges(G_tfidf_plot, pos, width=weights, alpha=0.4)
    nx.draw_networkx_labels(G_tfidf_plot, pos, font_size=7)

    plt.title("TF-IDF-Ähnlichkeitsnetzwerk der Briefe")
    plt.axis("off")
    plt.show()
else:
    print("Kein Netzwerk sichtbar. Probiere einen kleineren threshold.")

## 5. Teil B – Personen-Netzwerk

Jetzt bauen wir ein zweites Netzwerk, diesmal **zwischen Personen**.

Wir verwenden **nur** Briefe,

- bei denen `sender` vorhanden ist
- bei denen `recipient` vorhanden ist
- die **datiert** sind

Jeder Brief wird dann zu einer gerichteten Kante:

`sender -> recipient`

In [ ]:
person_df = df.copy()

# nur Briefe mit Sender, Empfänger und Jahr
person_df = person_df.dropna(subset=["sender", "recipient", "year_simple"]).copy()

person_df["sender"] = person_df["sender"].astype(str).str.strip()
person_df["recipient"] = person_df["recipient"].astype(str).str.strip()

person_df = person_df[(person_df["sender"] != "") & (person_df["recipient"] != "")].copy()
person_df["year_simple"] = person_df["year_simple"].astype(int)

print("Briefe im Personen-Netzwerk:", len(person_df))
person_df[[c for c in ["doc_id", "sender", "recipient", "year_simple", "dominant_topic"] if c in person_df.columns]].head(10)

### Kanten aggregieren

Mehrere Briefe zwischen denselben Personen werden zusammengezählt.

In [ ]:
edge_df = (
    person_df.groupby(["sender", "recipient"])
    .size()
    .reset_index(name="weight")
    .sort_values("weight", ascending=False)
)

edge_df.head(15)

In [ ]:
G_people = nx.DiGraph()

for _, row in edge_df.iterrows():
    G_people.add_edge(row["sender"], row["recipient"], weight=row["weight"])

print("Personen:", G_people.number_of_nodes())
print("Beziehungen:", G_people.number_of_edges())

In [ ]:
# Einfache Zentralitätsmasse
in_deg = dict(G_people.in_degree())
out_deg = dict(G_people.out_degree())
deg_total = dict(G_people.degree())

people_stats = pd.DataFrame({
    "person": list(G_people.nodes()),
    "degree_total": [deg_total[p] for p in G_people.nodes()],
    "degree_in": [in_deg[p] for p in G_people.nodes()],
    "degree_out": [out_deg[p] for p in G_people.nodes()],
}).sort_values("degree_total", ascending=False)

people_stats.head(15)

In [ ]:
# Für die Grafik nur die wichtigsten Personen zeigen
top_people = people_stats.head(20)["person"].tolist()
G_people_plot = G_people.subgraph(top_people).copy()

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G_people_plot, seed=42)

weights = [G_people_plot[u][v]["weight"] for u, v in G_people_plot.edges()]
node_sizes = [300 + 120 * G_people_plot.degree(n) for n in G_people_plot.nodes()]

nx.draw_networkx_nodes(G_people_plot, pos, node_size=node_sizes, alpha=0.85)
nx.draw_networkx_edges(
    G_people_plot,
    pos,
    arrows=True,
    arrowstyle="->",
    arrowsize=14,
    width=weights,
    alpha=0.45
)
nx.draw_networkx_labels(G_people_plot, pos, font_size=9)

plt.title("Personen-Netzwerk (Top-Personen)")
plt.axis("off")
plt.show()

## 6. Briefe als zeitliche Kanten-Tabelle

Für die zeitliche Analyse brauchen wir die Briefe **nicht** sofort als Bild, sondern zuerst als saubere **Ereignistabelle**:

- `sender`
- `recipient`
- `year_simple`
- optional: `topic`

In [ ]:
temporal_edges = person_df[[c for c in ["doc_id", "sender", "recipient", "year_simple", "dominant_topic", "topic_prob", "topic_keywords"] if c in person_df.columns]].copy()
temporal_edges = temporal_edges.sort_values("year_simple").reset_index(drop=True)

temporal_edges.head(15)

## 7. Entwicklung über die Zeit – ganz einfach mit pandas

Bevor wir `pathpyG` verwenden, machen wir zwei ganz einfache Auswertungen:

1. Wie viele Briefe gibt es pro Jahr?
2. Welche Topics treten pro Jahr auf?

In [ ]:
letters_per_year = (
    temporal_edges.groupby("year_simple")
    .size()
    .reset_index(name="n_letters")
    .sort_values("year_simple")
)

letters_per_year.head(15)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(letters_per_year["year_simple"], letters_per_year["n_letters"], marker="o")
plt.title("Datierte Briefe pro Jahr")
plt.xlabel("Jahr")
plt.ylabel("Anzahl Briefe")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
topic_year_df = temporal_edges.dropna(subset=["dominant_topic"]).copy()

if len(topic_year_df) > 0:
    topic_counts = (
        topic_year_df.groupby(["year_simple", "dominant_topic"])
        .size()
        .reset_index(name="n")
        .sort_values(["year_simple", "dominant_topic"])
    )
    topic_counts.head(15)
else:
    print("Keine Topics vorhanden.")

In [ ]:
if "topic_counts" in globals() and len(topic_counts) > 0:
    top_topics = (
        topic_counts.groupby("dominant_topic")["n"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .index
        .tolist()
    )

    plot_df = topic_counts[topic_counts["dominant_topic"].isin(top_topics)].copy()

    plt.figure(figsize=(12, 6))
    for topic_id in sorted(plot_df["dominant_topic"].unique()):
        sub = plot_df[plot_df["dominant_topic"] == topic_id]
        plt.plot(sub["year_simple"], sub["n"], marker="o", label=f"Topic {int(topic_id)}")

    plt.title("Häufige Topics über die Zeit")
    plt.xlabel("Jahr")
    plt.ylabel("Anzahl Briefe")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Kein Topic-Zeitplot möglich.")

## 8. Netzwerke in Zeitfenstern

Oft ist es anschaulicher, nicht jedes einzelne Jahr isoliert zu betrachten, sondern kleine **Zeitfenster** zu bilden.

Hier verwenden wir 5-Jahres-Fenster.

In [ ]:
window_size = 5

min_year = int(temporal_edges["year_simple"].min())
max_year = int(temporal_edges["year_simple"].max())

windows = []
start = min_year

while start <= max_year:
    end = start + window_size - 1
    windows.append((start, end))
    start += window_size

windows[:10]

In [ ]:
window_rows = []

for start, end in windows:
    sub = temporal_edges[(temporal_edges["year_simple"] >= start) & (temporal_edges["year_simple"] <= end)].copy()

    n_letters = len(sub)
    n_persons = len(set(sub["sender"]).union(set(sub["recipient"])))
    n_edges = len(sub.groupby(["sender", "recipient"]))

    # häufigstes Topic im Fenster
    if "dominant_topic" in sub.columns and sub["dominant_topic"].notna().any():
        top_topic = sub["dominant_topic"].value_counts().idxmax()
    else:
        top_topic = np.nan

    window_rows.append({
        "start": start,
        "end": end,
        "n_letters": n_letters,
        "n_persons": n_persons,
        "n_edges": n_edges,
        "top_topic": top_topic
    })

window_df = pd.DataFrame(window_rows)
window_df.head(10)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(window_df["start"], window_df["n_edges"], marker="o", label="Kanten")
plt.plot(window_df["start"], window_df["n_persons"], marker="o", label="Personen")
plt.title("Personen-Netzwerk in 5-Jahres-Fenstern")
plt.xlabel("Fensterbeginn")
plt.ylabel("Anzahl")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Optional: pathpyG für ein echtes zeitliches Netzwerk

`pathpyG` ist für **zeitgestempelte Kanten** gemacht.  
Wir erstellen darum aus jeder Briefbeziehung ein Tupel:

`(sender, recipient, year_simple)`

Wenn `pathpyG` nicht installiert ist, kannst du diesen Teil einfach überspringen.

In [ ]:
try:
    import pathpyG as pp
    HAS_PATHPY = True
    print("pathpyG wurde erfolgreich geladen.")
except Exception as e:
    HAS_PATHPY = False
    print("pathpyG ist nicht installiert oder konnte nicht geladen werden.")
    print(e)

In [ ]:
if HAS_PATHPY:
    tedges = list(
        temporal_edges[["sender", "recipient", "year_simple", "topic_keywords"]]
        .itertuples(index=False, name=None)
    )

    T = pp.TemporalGraph.from_edge_list(tedges)
    print(T)
    print("Knoten:", T.n)
    print("Ereignisse:", T.m)

### Gesamtes zeitlich aggregiertes Netzwerk

`pathpyG` kann das zeitliche Netzwerk auch als **statisches gewichtetes Netzwerk** zusammenfassen.

In [ ]:
if HAS_PATHPY:
    G_static = T.to_static_graph(weighted=True)
    print(G_static)

### Ein Zeitfenster auswählen

So können wir nur einen Ausschnitt betrachten.

In [ ]:
if HAS_PATHPY:
    start_year = int(temporal_edges["year_simple"].min())
    end_year = start_year + 25

    T_window = T.get_window(start_year, end_year)
    print(T_window)

### Rolling Time Windows

Das ist besonders praktisch, wenn wir die Entwicklung Schritt für Schritt verfolgen wollen.

In [ ]:
if HAS_PATHPY:
    rolling_rows = []

    roller = pp.algorithms.RollingTimeWindow(
        T,
        window_size=5,
        step_size=1,
        return_window=True
    )

    for g, w in roller:
        n_nodes = len(g.nodes) if hasattr(g, "nodes") else np.nan
        n_edges = len(g.edges) if hasattr(g, "edges") else np.nan

        rolling_rows.append({
            "window_start": w[0],
            "window_end": w[1],
            "n_nodes": n_nodes,
            "n_edges": n_edges
        })

    rolling_df = pd.DataFrame(rolling_rows)
    display(rolling_df.head(10))

In [ ]:
if HAS_PATHPY and len(rolling_df) > 0:
    plt.figure(figsize=(12, 5))
    plt.plot(rolling_df["window_start"], rolling_df["n_edges"], marker="o")
    plt.title("Rolling windows mit pathpyG")
    plt.xlabel("Fensterbeginn")
    plt.ylabel("Anzahl Kanten")
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
years = sorted(temporal_edges["year_simple"].dropna().astype(int).unique())
cumulative_graphs = []

for year in years:
    edges_until_year = list(
        temporal_edges.loc[
            temporal_edges["year_simple"] <= year,
            ["sender", "recipient", "year_simple"]
        ].itertuples(index=False, name=None)
    )

    T_year = pp.TemporalGraph.from_edge_list(edges_until_year)
    G_year = T_year.to_static_graph(weighted=True)

    cumulative_graphs.append((year, T_year, G_year))

print("Anzahl kumulativer Schritte:", len(cumulative_graphs))

In [ ]:
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(cumulative_graphs) - 1,
    step=1,
    description="Index"
)

def show_cumulative(i):
    year, T_year, G_year = cumulative_graphs[i]

    print(f"Netzwerk bis Jahr: {year}")
    print("Knoten:", G_year.n)
    print("Kanten:", G_year.m)

    pp.plot(G_year)

widgets.interact(show_cumulative, i=slider);

## 10. Topics an das Personen-Netzwerk knüpfen

Für viele historische Fragen ist spannend:

- **Wer** schreibt wem?
- **Wann**?
- **Zu welchem Thema**?

Darum schauen wir uns jetzt das **wahrscheinlichste Topic pro Beziehung** an.

In [ ]:
# Farben fuer Topics
topic_values = sorted(temporal_edges["dominant_topic"].dropna().unique())

cmap = plt.cm.get_cmap("tab10", len(topic_values) if len(topic_values) > 0 else 1)
topic_color_map = {
    topic: mcolors.to_hex(cmap(i))
    for i, topic in enumerate(topic_values)
}
default_color = "#cccccc"

years = sorted(temporal_edges["year_simple"].dropna().astype(int).unique())
cumulative_graphs = []

for year in years:
    df_until_year = temporal_edges[temporal_edges["year_simple"] <= year].copy()

    # temporaler Graph bis zu diesem Jahr
    edges_until_year = list(
        df_until_year[["sender", "recipient", "year_simple"]]
        .itertuples(index=False, name=None)
    )

    T_year = pp.TemporalGraph.from_edge_list(edges_until_year)
    G_year = T_year.to_static_graph(weighted=True)

    # haeufigstes Topic pro Beziehung bis zu diesem Jahr
    edge_topic_map = {}
    for (s, r), group in df_until_year.groupby(["sender", "recipient"]):
        topics = group["dominant_topic"].dropna().tolist()
        if len(topics) > 0:
            edge_topic_map[(s, r)] = Counter(topics).most_common(1)[0][0]

    # Farben in derselben Reihenfolge wie die Kanten in G_year
    edge_colors = []
    for e in G_year.edges:
        s = e[0]
        r = e[1]
        topic = edge_topic_map.get((s, r), None)
        edge_colors.append(topic_color_map.get(topic, default_color))

    cumulative_graphs.append((year, T_year, G_year, edge_colors, edge_topic_map))

print("Anzahl kumulativer Schritte:", len(cumulative_graphs))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

topic_label_map = {}
if "dominant_topic" in temporal_edges.columns and "topic_keywords" in temporal_edges.columns:
    tmp = (
        temporal_edges[["dominant_topic", "topic_keywords"]]
        .dropna()
        .drop_duplicates()
    )
    topic_label_map = dict(zip(tmp["dominant_topic"], tmp["topic_keywords"]))

def shorten(s, n=50):
    if not isinstance(s, str):
        return ""
    return s[:n] + "..." if len(s) > n else s

year_values = [int(item[0]) for item in cumulative_graphs]

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(year_values) - 1,
    step=1,
    description="Jahr",
    continuous_update=False
)

out = widgets.Output()

def make_html_legend(current_topics):
    rows = []
    for t in current_topics:
        if t in topic_color_map:
            color = topic_color_map[t]
            label = f"{t}: {shorten(topic_label_map.get(t, ''))}"
            rows.append(
                f"""
                <div style="display:flex; align-items:center; margin:4px 0;">
                    <div style="
                        width:18px;
                        height:18px;
                        background:{color};
                        border:1px solid #999;
                        margin-right:8px;
                    "></div>
                    <div>{label}</div>
                </div>
                """
            )
    return "<div style='margin-top:10px;'>" + "".join(rows) + "</div>"

def show_cumulative(change=None):
    i = slider.value
    year, T_year, G_year, edge_colors, edge_topic_map = cumulative_graphs[i]

    with out:
        clear_output(wait=True)

        print(f"Netzwerk bis Jahr: {year}")
        print("Knoten:", len(G_year.nodes))
        print("Kanten:", len(G_year.edges))

        # interaktiver pathpy-Plot ohne matplotlib
        # dadurch wird die Visualisierung wieder dynamischer
        pp.plot(G_year, edge_color=edge_colors)

        # HTML-Legende mit echten Farben + Keywords
        current_topics = sorted(set(edge_topic_map.values()))
        display(HTML(make_html_legend(current_topics)))

slider.observe(show_cumulative, names="value")
display(slider, out)
show_cumulative()

## 11. Ein kleines Beispielnetz für ein einzelnes Topic

So kann man sehen, welche Personen besonders stark mit einem bestimmten Thema verbunden sind.

In [ ]:
if "relation_topic" in globals() and len(relation_topic) > 0:
    selected_topic = relation_topic["dominant_topic"].value_counts().index[0]
    sub = relation_topic[relation_topic["dominant_topic"] == selected_topic].copy()

    G_topic = nx.DiGraph()
    for _, row in sub.iterrows():
        G_topic.add_edge(row["sender"], row["recipient"], weight=row["n"])

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G_topic, seed=42)
    weights = [G_topic[u][v]["weight"] for u, v in G_topic.edges()]

    nx.draw_networkx_nodes(G_topic, pos, node_size=700, alpha=0.85)
    nx.draw_networkx_edges(G_topic, pos, width=weights, arrows=True, alpha=0.45)
    nx.draw_networkx_labels(G_topic, pos, font_size=9)

    plt.title(f"Personen-Netzwerk für Topic {int(selected_topic)}")
    plt.axis("off")
    plt.show()
else:
    print("Kein Topic-Netzwerk möglich.")

## 12. Ergebnisse speichern

Wir speichern die wichtigsten Zwischenergebnisse als CSV.

In [ ]:
network_dir = base_dir / "network_analysis"
network_dir.mkdir(parents=True, exist_ok=True)

deg_df.to_csv(network_dir / "tfidf_network_degrees.csv", index=False)
edge_df.to_csv(network_dir / "person_network_edges.csv", index=False)
people_stats.to_csv(network_dir / "person_network_node_stats.csv", index=False)
temporal_edges.to_csv(network_dir / "temporal_edges.csv", index=False)
window_df.to_csv(network_dir / "time_windows_summary.csv", index=False)

if "relation_topic" in globals():
    relation_topic.to_csv(network_dir / "relation_topic_counts.csv", index=False)

sorted(network_dir.glob("*.csv"))

## 13. Mini-Übungen

1. Verändere im TF-IDF-Netzwerk den `threshold`.  
   Wie verändert sich die Zahl der Kanten?

2. Begrenze das TF-IDF-Netzwerk einmal auf nur ein Subkorpus.  
   Wirkt das Netzwerk dichter oder dünner?

3. Verändere beim Personen-Netzwerk die Fenstergrösse von 5 auf 10 Jahre.  
   Was passiert?

4. Suche im Personen-Netzwerk nach Beziehungen, die über mehrere Zeitfenster hinweg bestehen.

5. Wenn Topics vorhanden sind:  
   Welches Topic ist in welcher Phase besonders sichtbar?

## 14. Methodische Reflexion

Ein paar wichtige Punkte:

- Ein **TF-IDF-Netzwerk** zeigt **textuelle Ähnlichkeit**, nicht automatisch historische Nähe.
- Ein **Personen-Netzwerk** ist immer nur so gut wie die Qualität von `sender`, `recipient` und `date_when`.
- Die Datierung antiker Briefe ist oft unsicher. Darum ist eine einfache Jahreszahl immer nur eine **Arbeitsheuristik**.
- Topics helfen bei der Interpretation, sind aber **modellierte Kategorien** und keine objektiven Tatsachen.